# **Library**

Instalasi dan impor seluruh library yang digunakan dalam pipeline ini, mencakup:
- `yfinance` untuk mengambil data harga saham historis
- `requests` & `pandas` untuk scraping dan pengolahan data laporan keuangan
- `supabase` untuk koneksi dan upload data ke database
- `numpy`, `re`, `io`, `json` untuk kebutuhan komputasi dan parsing data

In [ ]:
!pip install supabase

In [ ]:
import json
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from datetime import datetime
from supabase import create_client, Client
import io
import re

# **Load Supabase**

Inisialisasi koneksi ke database Supabase menggunakan URL dan API Key.
Ganti `your_supabase_url` dan `your_supabase_key` dengan kredensial proyek Supabase Anda.
Fungsi `upload_df` digunakan untuk mengunggah DataFrame ke tabel Supabase
menggunakan metode upsert secara batch, sehingga pipeline aman dijalankan berulang kali tanpa menghasilkan duplikasi data.

In [ ]:
SUPABASE_URL = 'your_supabase_url.supabase.co'
SUPABASE_KEY = 'your_supabase_key'

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Koneksi berhasil!')

Koneksi berhasil!


In [ ]:
def upload_df(supabase: Client, table_name: str, df: pd.DataFrame,
              date_col: str = 'date', batch_size: int = 500):
    """
    Upload DataFrame ke satu tabel Supabase menggunakan metode upsert.
    - date_col  : nama kolom tanggal yang akan dikonversi ke format string ISO
    - Proses upsert memastikan pipeline aman dijalankan berulang kali tanpa duplikasi data
    """
    upload_df = df.copy()

    # Konversi date ke string ISO
    if date_col in upload_df.columns:
        upload_df[date_col] = pd.to_datetime(upload_df[date_col]).dt.strftime('%Y-%m-%d')

    # Mengganti nilai NaN dengan None agar kompatibel dengan format JSON
    upload_df = upload_df.where(pd.notnull(upload_df), None)

    records     = upload_df.to_dict(orient='records')
    total       = len(records)
    success     = 0

    for i in range(0, total, batch_size):
        batch = records[i:i + batch_size]
        try:
            supabase.table(table_name).upsert(batch).execute()
            success += len(batch)
        except Exception as e:
            print(f'  [ERROR] {table_name} batch {i}–{i + len(batch)}: {e}')

    status = 'OK' if success == total else 'PARTIAL'
    print(f'  [{status}] {table_name}: {success}/{total} rows')
    return success

# **Ticker Config**

Memuat daftar 45 saham LQ45 dari file konfigurasi `tickers.json`.
File ini menyimpan kode ticker beserta suffix bursa (`.JK`) yang digunakan
oleh Yahoo Finance untuk mengidentifikasi saham yang terdaftar di Bursa Efek Indonesia.

In [ ]:
# Load ticker list dari file JSON
with open('tickers.json', 'r') as f:
    ticker_config = json.load(f)

suffix     = ticker_config['suffix']              # '.JK'
tickers    = ticker_config['tickers']             # list tanpa suffix
tickers_yf = [t + suffix for t in tickers]        # format yfinance

print(f'Total ticker: {len(tickers_yf)}')
print(tickers_yf)

Total ticker: 45
['AADI.JK', 'ADMR.JK', 'ADRO.JK', 'AKRA.JK', 'AMMN.JK', 'AMRT.JK', 'ANTM.JK', 'ASII.JK', 'BBCA.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BMRI.JK', 'BREN.JK', 'BRIS.JK', 'BRPT.JK', 'CPIN.JK', 'CTRA.JK', 'EXCL.JK', 'GOTO.JK', 'ICBP.JK', 'INCO.JK', 'INDF.JK', 'INKP.JK', 'ISAT.JK', 'ITMG.JK', 'JPFA.JK', 'JSMR.JK', 'KLBF.JK', 'MAPA.JK', 'MAPI.JK', 'MBMA.JK', 'MDKA.JK', 'MEDC.JK', 'NCKL.JK', 'PGAS.JK', 'PGEO.JK', 'PTBA.JK', 'SCMA.JK', 'SMGR.JK', 'TLKM.JK', 'TOWR.JK', 'UNTR.JK', 'UNVR.JK', 'ARTO.JK']


# **Teknikal**

Pengambilan data harga historis dan perhitungan indikator teknikal untuk seluruh saham LQ45 sejak tahun 2016. Indikator yang dihitung meliputi:
- Trend: SMA 5, SMA 20
- Momentum: RSI 14, MACD, Stochastic (K & D)
- Volatilitas: Bollinger Bands, ATR 14
- Volume: OBV
- Return: return_1d, return_3d, return_5d

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def bollinger_bands(series, period=20, std_dev=2):
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = sma + (std * std_dev)
    lower = sma - (std * std_dev)
    return upper, lower

def macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast).mean()
    ema_slow = series.ewm(span=slow).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal).mean()
    return macd_line, signal_line

def compute_atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def compute_obv(close, volume):
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()

def compute_stochastic(high, low, close, k_period=14, d_period=3):
    lowest_low   = low.rolling(k_period).min()
    highest_high = high.rolling(k_period).max()
    k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    d = k.rolling(d_period).mean()
    return k, d

In [ ]:
technical = {}

for ticker in tickers_yf:
    data = yf.download(ticker, start='2016-01-01',
                       end=datetime.now().strftime('%Y-%m-%d'), progress=False)
    if data.empty:
        print(f'[SKIP] {ticker} - no data')
        continue

    # Flatten MultiIndex columns
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data.loc[:, ~data.columns.duplicated()]

    data['SMA_5']         = data['Close'].rolling(5).mean()
    data['SMA_20']        = data['Close'].rolling(20).mean()
    data['RSI_14']        = compute_rsi(data['Close'], 14)
    data['BB_upper'], data['BB_lower'] = bollinger_bands(data['Close'], 20, 2)
    data['MACD'], data['MACD_signal']  = macd(data['Close'])

    data['ATR_14']        = compute_atr(data['High'], data['Low'], data['Close'], 14)
    data['OBV']           = compute_obv(data['Close'], data['Volume'])
    data['Stoch_K'], data['Stoch_D'] = compute_stochastic(data['High'], data['Low'], data['Close'])
    data['return_1d']     = data['Close'].pct_change(1)
    data['return_3d']     = data['Close'].pct_change(3)
    data['return_5d']     = data['Close'].pct_change(5)

    data.dropna(inplace=True)

    df = data.reset_index()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.loc[:, ~df.columns.duplicated()]

    df['ticker'] = ticker.replace('.JK', '')
    ticker_code  = ticker.replace('.JK', '')
    technical[ticker_code] = df
    print(f'[OK] {ticker}: {len(df)} rows')

print(f'\nTotal ticker berhasil: {len(technical)}')
print(f'Contoh kolom ({list(technical.keys())[0]}): {list(technical[list(technical.keys())[0]].columns)}')

/tmp/ipykernel_20820/3365656588.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2016-01-01',


[OK] AADI.JK: 278 rows


/tmp/ipykernel_20820/3365656588.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2016-01-01',


[OK] ADMR.JK: 984 rows


/tmp/ipykernel_20820/3365656588.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2016-01-01',


...

[OK] UNTR.JK: 2491 rows


/tmp/ipykernel_20820/3365656588.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2016-01-01',


[OK] UNVR.JK: 2492 rows


/tmp/ipykernel_20820/3365656588.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2016-01-01',


[OK] ARTO.JK: 2453 rows

Total ticker berhasil: 45
Contoh kolom (AADI): ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_5', 'SMA_20', 'RSI_14', 'BB_upper', 'BB_lower', 'MACD', 'MACD_signal', 'ATR_14', 'OBV', 'Stoch_K', 'Stoch_D', 'return_1d', 'return_3d', 'return_5d', 'ticker']


# **Upload Teknikal ke Supabase**

Upload hasil perhitungan data teknikal ke Supabase. Setiap ticker disimpan
pada tabel terpisah dengan format penamaan `{ticker_code}_technical`.
Proses dilakukan secara batch dengan ukuran 500 baris per pengiriman untuk
menghindari timeout.

In [ ]:
col_map_technical = {
    'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low',
    'Close': 'close', 'Volume': 'volume',
    'SMA_5': 'sma_5', 'SMA_20': 'sma_20',
    'RSI_14': 'rsi_14',
    'BB_upper': 'bb_upper', 'BB_lower': 'bb_lower',
    'MACD': 'macd', 'MACD_signal': 'macd_signal',
    'ATR_14': 'atr_14', 'OBV': 'obv',
    'Stoch_K': 'stoch_k', 'Stoch_D': 'stoch_d',
    'return_1d': 'return_1d', 'return_3d': 'return_3d', 'return_5d': 'return_5d',
}

results_technical = {}

for ticker_code, df in technical.items():
    table_name  = f'{ticker_code.lower()}_technical'
    existing    = {k: v for k, v in col_map_technical.items() if k in df.columns}
    df_upload   = df[list(existing.keys())].rename(columns=existing)

    results_technical[ticker_code] = upload_df(supabase, table_name, df_upload, date_col='date')

print(f'Ticker   : {len(results_technical)}/{len(tickers)}')
print(f'Total rows: {sum(results_technical.values())}')

  [OK] aadi_technical: 278/278 rows
  [OK] admr_technical: 984/984 rows
  [OK] adro_technical: 2491/2491 rows
  [OK] akra_technical: 2492/2492 rows
  [OK] ammn_technical: 620/620 rows
  [OK] amrt_technical: 2492/2492 rows
  [OK] antm_technical: 2492/2492 rows
  [OK] asii_technical: 2491/2491 rows
  [OK] bbca_technical: 2492/2492 rows
  [OK] bbni_technical: 2492/2492 rows
  [OK] bbri_technical: 2492/2492 rows
  [OK] bbtn_technical: 2492/2492 rows
  [OK] bmri_technical: 2492/2492 rows
  [OK] bren_technical: 557/557 rows
  [OK] bris_technical: 1899/1899 rows
  [OK] brpt_technical: 2492/2492 rows
  [OK] cpin_technical: 2491/2491 rows
  [OK] ctra_technical: 2491/2491 rows
  [OK] excl_technical: 2492/2492 rows
  [OK] goto_technical: 915/915 rows
  [OK] icbp_technical: 2491/2491 rows
  [OK] inco_technical: 2491/2491 rows
  [OK] indf_technical: 2492/2492 rows
  [OK] inkp_technical: 2491/2491 rows
  [OK] isat_technical: 2492/2492 rows
  [OK] itmg_technical: 2492/2492 rows
  [OK] jpfa_technical:

# **Fundamental**

Pengambilan data laporan keuangan kuartalan untuk seluruh saham LQ45 melalui
scraping dari stockanalysis.com. Data yang diambil mencakup balance sheet dan income statement, kemudian diolah menjadi metrik seperti ROA, pertumbuhan revenue, dan net income (QoQ & YoY).
Bagi emiten yang melaporkan dalam USD, nilai finansial dikonversi ke IDR
menggunakan data kurs historis USD/IDR dari Yahoo Finance.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
}

# Mengambil data historis kurs USD/IDR sejak tahun 2023 sebagai referensi konversi mata uang
fx = yf.download('USDIDR=X', start='2023-01-01',
                 end=datetime.now().strftime('%Y-%m-%d'), progress=False)
if isinstance(fx.columns, pd.MultiIndex):
    fx.columns = fx.columns.get_level_values(0)
fx_daily = fx[['Close']].rename(columns={'Close': 'usd_idr'})
fx_daily.index = pd.to_datetime(fx_daily.index)
fx_sorted = fx_daily.reset_index().rename(columns={'Date': 'date'}).sort_values('date')
print(f'Kurs USD/IDR tersedia: {fx_sorted["date"].min().date()} s/d {fx_sorted["date"].max().date()}')

def get_financial_meta(html: str) -> dict:
    match = re.search(
        r'[Ff]inancials?\s+in\s+(millions?|billions?|thousands?)?\s*(IDR|USD|SGD|EUR)',
        html
    )
    if match:
        unit     = match.group(1).lower().rstrip('s') if match.group(1) else 'raw'
        currency = match.group(2).upper()
        return {'currency': currency, 'unit': unit}
    return {'currency': 'unknown', 'unit': 'unknown'}

UNIT_MULTIPLIER = {
    'million'  : 1_000_000,
    'billion'  : 1_000_000_000,
    'thousand' : 1_000,
    'raw'      : 1,
    'unknown'  : 1,
}

def fetch_table_with_meta(url: str) -> tuple:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    tables = pd.read_html(io.StringIO(r.text))
    if not tables:
        raise ValueError(f'No tables found at {url}')
    meta = get_financial_meta(r.text)
    return tables[0], meta

def _index_to_datetime(idx) -> pd.DatetimeIndex:
    if isinstance(idx, pd.MultiIndex):
        base = idx.get_level_values(-1).astype(str)
    else:
        base = idx.astype(str)
    extracted = pd.Series(base, index=idx).str.extract(r'([A-Za-z]{3}\s+\d{1,2},\s+\d{4})', expand=False)
    cleaned = extracted.fillna(pd.Series(base, index=extracted.index))
    return pd.to_datetime(cleaned, errors='coerce')

def table_to_timeseries(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    first_col = df.columns[0]
    df = df.set_index(first_col).T
    dt_idx = _index_to_datetime(df.index)
    keep = ~dt_idx.isna()
    df = df.loc[keep].copy()
    df.index = dt_idx[keep]
    df.index.name = 'date'
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.replace(r'_$', '', regex=True)
    )
    def to_number(x):
        if pd.isna(x): return pd.NA
        s = str(x).strip()
        if s in {'-', '—', ''}: return pd.NA
        neg = s.startswith('(') and s.endswith(')')
        s = s.replace(',', '').replace('(', '').replace(')', '')
        mult = 1.0
        if s.endswith('B'): mult = 1e9; s = s[:-1]
        elif s.endswith('M'): mult = 1e6; s = s[:-1]
        try:
            val = float(s) * mult
            return -val if neg else val
        except: return pd.NA
    for c in df.columns:
        df[c] = df[c].map(to_number)
    return df

def pick_col(df: pd.DataFrame, candidates: list) -> pd.Series:
    for c in candidates:
        if c in df.columns: return df[c]
    for c in df.columns:
        for k in candidates:
            if k in c: return df[c]
    return pd.Series([pd.NA] * len(df), index=df.index)

def compute_growth(series: pd.Series, periods: int) -> pd.Series:
    return series.replace(pd.NA, np.nan).astype('float64').pct_change(periods=periods)

/tmp/ipykernel_20820/3486855255.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  fx = yf.download('USDIDR=X', start='2023-01-01',


Kurs USD/IDR tersedia: 2023-01-02 s/d 2026-03-10


In [ ]:
fundamental = {}

fin_cols = ['total_assets', 'total_liabilities', 'short_term_debt',
            'long_term_debt', 'revenue', 'net_income']

for t in tickers:
    url_bs = f'https://stockanalysis.com/quote/idx/{t}/financials/balance-sheet/?p=quarterly'
    url_is = f'https://stockanalysis.com/quote/idx/{t}/financials/?p=quarterly'
    try:
        bs_raw, bs_meta = fetch_table_with_meta(url_bs)
        is_raw, is_meta = fetch_table_with_meta(url_is)
        bs  = table_to_timeseries(bs_raw)
        inc = table_to_timeseries(is_raw)
    except Exception as e:
        print(f'[SKIP] {t}: {e}')
        continue

    bs_mult  = UNIT_MULTIPLIER.get(bs_meta['unit'], 1)
    is_mult  = UNIT_MULTIPLIER.get(is_meta['unit'], 1)
    currency = is_meta['currency']

    df_f = pd.DataFrame(index=bs.index.union(inc.index)).sort_index()
    df_f.index.name = 'date'

    df_f['total_assets']      = pick_col(bs,  ['total_assets'])                                                  * bs_mult
    df_f['total_liabilities'] = pick_col(bs,  ['total_liabilities'])                                             * bs_mult
    df_f['short_term_debt']   = pick_col(bs,  ['short_term_debt', 'current_debt', 'short_long_term_debt'])       * bs_mult
    df_f['long_term_debt']    = pick_col(bs,  ['long_term_debt', 'long_term_debt_and_capital_lease_obligation'])  * bs_mult
    df_f['revenue']           = pick_col(inc, ['revenue', 'total_revenue'])                                      * is_mult
    df_f['net_income']        = pick_col(inc, ['net_income', 'net_income_common_stockholders', 'net_income_to_common']) * is_mult

    # Mengonversi nilai finansial dari USD ke IDR menggunakan kurs pada tanggal laporan terdekat
    if currency == 'USD':
        df_temp = df_f[fin_cols].reset_index()  # index = date
        df_temp = df_temp.rename(columns={'date': 'date'}).sort_values('date')

        # Menggunakan merge_asof untuk mencocokkan setiap tanggal laporan dengan kurs terdekat yang tersedia
        df_temp = pd.merge_asof(df_temp, fx_sorted, on='date', direction='nearest')

        for col in fin_cols:
            df_temp[col] = df_temp[col] * df_temp['usd_idr']

        df_temp = df_temp.drop(columns=['usd_idr']).set_index('date')
        df_f[fin_cols] = df_temp[fin_cols]

    df_f['roa'] = df_f['net_income'] / df_f['total_assets']

    df_f['revenue_qoq']    = compute_growth(df_f['revenue'],    periods=1)
    df_f['net_income_qoq'] = compute_growth(df_f['net_income'], periods=1)
    df_f['revenue_yoy']    = compute_growth(df_f['revenue'],    periods=4)
    df_f['net_income_yoy'] = compute_growth(df_f['net_income'], periods=4)

    df_f = df_f[df_f.index >= pd.Timestamp('2024-01-01')].copy()

    core_cols = ['total_assets', 'total_liabilities', 'short_term_debt',
                 'long_term_debt', 'revenue', 'net_income', 'roa']
    qoq_cols  = ['revenue_qoq', 'net_income_qoq']
    yoy_cols  = ['revenue_yoy', 'net_income_yoy']

    df_f['fund_available'] = df_f[core_cols].notna().all(axis=1).astype(int)
    df_f['qoq_available']  = df_f[qoq_cols].notna().all(axis=1).astype(int)
    df_f['yoy_available']  = df_f[yoy_cols].notna().all(axis=1).astype(int)

    df_f = df_f.sort_index()
    df_f = df_f.ffill()
    df_f = df_f.fillna(0)

    df_f = df_f.reset_index()
    df_f['ticker'] = t
    fundamental[t] = df_f
    print(f'[OK] {t}: {len(df_f)} rows | {currency} ({is_meta["unit"]})')

print(f'\nTotal ticker berhasil: {len(fundamental)}')

/tmp/ipykernel_20820/1210773826.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_f = df_f.fillna(0)


[OK] AADI: 8 rows | USD (million)


/tmp/ipykernel_20820/1210773826.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_f = df_f.ffill()
/tmp/ipykernel_20820/1210773826.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_f = df_f.fillna(0)


[OK] ADMR: 8 rows | USD (million)


/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.replace(pd.NA, np.nan).astype('float64').pct_change(periods=periods)
/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return series.replace(pd.NA, np.nan).astype('float64').pct_change(periods=periods)
/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in 

...

/tmp/ipykernel_20820/1210773826.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_f = df_f.ffill()


[OK] TOWR: 7 rows | IDR (million)
[OK] UNTR: 8 rows | IDR (million)


/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.replace(pd.NA, np.nan).astype('float64').pct_change(periods=periods)
/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return series.replace(pd.NA, np.nan).astype('float64').pct_change(periods=periods)
/tmp/ipykernel_20820/3486855255.py:93: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in 

[OK] UNVR: 8 rows | IDR (million)
[OK] ARTO: 7 rows | IDR (million)

Total ticker berhasil: 45


/tmp/ipykernel_20820/1210773826.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_f = df_f.fillna(0)


# **Upload Fundamental ke Supabase**

Upload data fundamental hasil pengolahan ke Supabase. Setiap ticker disimpan
pada tabel terpisah dengan format penamaan `{ticker_code}_fundamental`.
Dibuat dictionary nama kolom sebelum upload untuk memastikan kesesuaian
dengan skema tabel yang telah didefinisikan di Supabase.

In [ ]:
col_map_fundamental = {
    'date'              : 'date',
    'total_assets'      : 'total_assets',
    'total_liabilities' : 'total_liabilities',
    'short_term_debt'   : 'short_term_debt',
    'long_term_debt'    : 'long_term_debt',
    'revenue'           : 'revenue',
    'net_income'        : 'net_income',
    'roa'               : 'roa',
    'revenue_qoq'       : 'revenue_qoq',
    'net_income_qoq'    : 'net_income_qoq',
    'revenue_yoy'       : 'revenue_yoy',
    'net_income_yoy'    : 'net_income_yoy',
    'fund_available'    : 'fund_available',
    'qoq_available'     : 'qoq_available',
    'yoy_available'     : 'yoy_available',
}

results_fundamental = {}

for ticker_code, df in fundamental.items():
    table_name = f'{ticker_code.lower()}_fundamental'
    existing   = {k: v for k, v in col_map_fundamental.items() if k in df.columns}
    df_upload  = df[list(existing.keys())].rename(columns=existing)

    results_fundamental[ticker_code] = upload_df(supabase, table_name, df_upload, date_col='date')

print(f'\nTicker    : {len(results_fundamental)}/{len(tickers)}')
print(f'Total rows: {sum(results_fundamental.values())}')

  [OK] aadi_fundamental: 8/8 rows
  [OK] admr_fundamental: 8/8 rows
  [OK] adro_fundamental: 8/8 rows
  [OK] akra_fundamental: 7/7 rows
  [OK] ammn_fundamental: 7/7 rows
  [OK] amrt_fundamental: 7/7 rows
  [OK] antm_fundamental: 7/7 rows
  [OK] asii_fundamental: 8/8 rows
  [OK] bbca_fundamental: 8/8 rows
  [OK] bbni_fundamental: 8/8 rows
  [OK] bbri_fundamental: 8/8 rows
  [OK] bbtn_fundamental: 8/8 rows
  [OK] bmri_fundamental: 8/8 rows
  [OK] bren_fundamental: 7/7 rows
  [OK] bris_fundamental: 8/8 rows
  [OK] brpt_fundamental: 7/7 rows
  [OK] cpin_fundamental: 7/7 rows
  [OK] ctra_fundamental: 7/7 rows
  [OK] excl_fundamental: 8/8 rows
  [OK] goto_fundamental: 7/7 rows
  [OK] icbp_fundamental: 7/7 rows
  [OK] inco_fundamental: 7/7 rows
  [OK] indf_fundamental: 7/7 rows
  [OK] inkp_fundamental: 7/7 rows
  [OK] isat_fundamental: 8/8 rows
  [OK] itmg_fundamental: 8/8 rows
  [OK] jpfa_fundamental: 8/8 rows
  [OK] jsmr_fundamental: 8/8 rows
  [OK] klbf_fundamental: 7/7 rows
  [OK] mapa_fu